[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_21/iris_neural_network.ipynb)

# Neuronale Netze mit dem Iris-Datensatz 🌸

**Ziel:** Klassifikation von Iris-Blüten mit einem neuronalen Netz

In diesem Notebook lernen wir:
- Wie man den klassischen Iris-Datensatz für neuronale Netze vorbereitet
- Aufbau und Training eines Multi-Layer Perceptrons (MLP)
- Evaluation und Interpretation der Ergebnisse
- Praktische Tipps für neuronale Netze bei kleinen Datensätzen

**Der Iris-Datensatz:** 150 Iris-Blüten, 4 Features (Kelch-/Blütenblattlänge und -breite), 3 Klassen (Setosa, Versicolor, Virginica)

## 1. Import Required Libraries

Wir benötigen verschiedene Bibliotheken für Datenverarbeitung, neuronale Netze und Visualisierung.

In [ ]:
# Grundlegende Bibliotheken
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn für Daten und Metriken
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Für bessere Plots
plt.style.use('default')
sns.set_palette("husl")

# Reproduzierbarkeit
np.random.seed(43)

print("✅ Alle Bibliotheken erfolgreich importiert!")

## 2. Load and Explore the Iris Dataset

Der Iris-Datensatz ist ein Klassiker des Machine Learning. Lass uns ihn laden und verstehen.

In [ ]:
# Iris-Datensatz laden
iris = load_iris()

# In DataFrame konvertieren für bessere Handhabung
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("📊 Iris-Datensatz Überblick:")
print(f"Shape: {df.shape}")
print(f"Features: {list(iris.feature_names)}")
print(f"Klassen: {list(iris.target_names)}")
print("\nErste 5 Zeilen:")
print(df.head())

In [ ]:
# Datenverteilung visualisieren
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Iris Dataset - Feature Distributions', fontsize=16, fontweight='bold')

# Pairplot-ähnliche Visualisierung der wichtigsten Features
features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
colors = ['red', 'green', 'blue']

for i, feature in enumerate(features):
    ax = axes[i//2, i%2]
    for j, species in enumerate(['setosa', 'versicolor', 'virginica']):
        data = df[df['species_name'] == species][feature]
        ax.hist(data, alpha=0.7, label=species, color=colors[j], bins=15)
    
    ax.set_title(f'{feature}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistische Summary
print("\n📈 Statistische Zusammenfassung:")
print(df.groupby('species_name')[features].mean().round(2))

## 3. Data Preprocessing and Splitting

Standardisierte Merkmale helfen bei diesem Training. Wir skalieren die Features und teilen die Daten auf.

In [ ]:
# Features (X) und Zielwerte (y) trennen
X = iris.data  # 4 Features: sepal length/width, petal length/width
y = iris.target  # 3 Klassen: 0=setosa, 1=versicolor, 2=virginica

print("🔍 Datenform vor Preprocessing:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Klassen: {np.unique(y)}")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n📊 Datenaufteilung:")
print(f"Training: {X_train.shape[0]} Samples")
print(f"Test: {X_test.shape[0]} Samples")

# Feature Scaling - WICHTIG für neuronale Netze!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n⚖️ Feature-Skalierung:")
print("Vor Skalierung (Training):")
print(f"Mean: {X_train.mean(axis=0).round(2)}")
print(f"Std:  {X_train.std(axis=0).round(2)}")
print("\nNach Skalierung (Training):")
print(f"Mean: {X_train_scaled.mean(axis=0).round(2)}")
print(f"Std:  {X_train_scaled.std(axis=0).round(2)}")

## 4. Build Neural Network Model

Jetzt erstellen wir unser neuronales Netz! Wir verwenden ein Multi-Layer Perceptron (MLP) mit zwei versteckten Schichten.

In [ ]:
# Neuronales Netz definieren
mlp = MLPClassifier(
    hidden_layer_sizes=(10, 5),    # Zwei versteckte Schichten: 10 und 5 Neuronen
    activation='relu',             # ReLU-Aktivierungsfunktion
    solver='adam',                 # Adam-Optimizer (modern und effizient)
    alpha=0.001,                   # L2-Regularisierung
    batch_size='auto',             # Automatische Batch-Größe
    learning_rate='constant',      # Dieser Parameter betrifft nur den SGD-Solver
    learning_rate_init=0.001,      # Anfängliche Lernrate
    max_iter=1000,                 # Maximale Iterationen
    shuffle=True,                  # Daten in jeder Epoche mischen
    random_state=42,               # Reproduzierbarkeit
    verbose=False                  # Kompakte Ausgabe, auch in der Cross-Validation
)

print("🧠 Neuronales Netz Architektur:")
print("Input Layer:    4 Neuronen (Features)")
print("Hidden Layer 1: 10 Neuronen (ReLU)")
print("Hidden Layer 2: 5 Neuronen (ReLU)")
print("Output Layer:   3 Neuronen (Softmax)")
print("\n🔧 Hyperparameter:")
print(f"Aktivierung: {mlp.activation}")
print(f"Optimizer: {mlp.solver}")
print(f"Lernrate: {mlp.learning_rate_init}")
print(f"Regularisierung: {mlp.alpha}")
print(f"Max. Iterationen: {mlp.max_iter}")

## 5. Train the Neural Network

Zeit zum Training! Das neuronale Netz lernt die Muster in den Iris-Daten.

In [ ]:
# Training starten
print("🚀 Training gestartet...")
import time
start_time = time.time()

# Model trainieren
mlp.fit(X_train_scaled, y_train)

training_time = time.time() - start_time
print(f"\n✅ Training abgeschlossen in {training_time:.2f} Sekunden")
print(f"📊 Anzahl Iterationen: {mlp.n_iter_}")
print(f"Training vor dem Iterationslimit beendet: {mlp.n_iter_ < mlp.max_iter}")
print("Das Abbruchkriterium prüft die Verbesserung des Trainingsfehlers, nicht die Testleistung.")

# Training-Loss visualisieren
plt.figure(figsize=(10, 6))
plt.plot(mlp.loss_curve_, 'b-', linewidth=2)
plt.title('Training Loss Curve', fontsize=14, fontweight='bold')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.show()

print(f"📉 Final Loss: {mlp.loss_curve_[-1]:.6f}")

## 5.5. Cross-Validation: Leistung auf mehreren Aufteilungen prüfen

Wir vergleichen 3-, 5-, 8- und 10-fache Cross-Validation auf den Trainingsdaten.
Die eigentlichen Testdaten bleiben dabei unbenutzt.

Alle Fold-Modelle verwenden dieselben Einstellungen wie das Hauptmodell.
Ihre gelernten Gewichte unterscheiden sich, weil sie unterschiedliche Daten sehen.
Der Scaler wird in jedem Fold nur auf dessen Trainingsdaten angepasst.

Wir zeigen Mittelwert und Standardabweichung der Accuracy. Die Streuung beschreibt
Unterschiede zwischen den Folds; sie ist kein Konfidenzintervall. Wenige Fehler können
die Accuracy kleiner Folds bereits deutlich verändern. Die kleinste Streuung in
einem Versuch beweist keine allgemein optimale Fold-Anzahl.

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import make_pipeline

# clone übernimmt die Einstellungen, aber keine bereits gelernten Gewichte.
mlp_cv = clone(mlp)
cv_pipeline = make_pipeline(StandardScaler(), mlp_cv)
cv_results = {}

print("Cross-Validation mit denselben Modelleinstellungen wie beim Hauptmodell")
print("Der Scaler wird in jedem Fold nur auf dessen Trainingsdaten angepasst.")

for n_folds in [3, 5, 8, 10]:
    folds = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    result = cross_validate(
        cv_pipeline, X_train, y_train, cv=folds,
        scoring='accuracy', return_estimator=True, n_jobs=1
    )
    scores = result['test_score']
    iterations = [model.named_steps['mlpclassifier'].n_iter_
                  for model in result['estimator']]
    cv_results[n_folds] = result
    print(f"\n{n_folds}-fach: Accuracy {scores.mean():.3f}, Streuung {scores.std():.3f}")
    print(f"Fold-Accuracies: {scores.round(3)}")
    print(f"Trainingsiterationen: {iterations} (Limit: {mlp.max_iter})")

# 5-fach CV dient als fester Referenzwert, nicht als Gewinner eines Rankings.
cv_scores = cv_results[5]['test_score']
cv_mean, cv_std = cv_scores.mean(), cv_scores.std()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(1, 6), cv_scores)
axes[0].set_title('5-fach CV: Accuracy pro Fold')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.05)

fold_counts = list(cv_results)
means = [cv_results[k]['test_score'].mean() for k in fold_counts]
stds = [cv_results[k]['test_score'].std() for k in fold_counts]
axes[1].errorbar(fold_counts, means, yerr=stds, fmt='o', capsize=5)
axes[1].set_title('Mittelwert und Streuung zwischen Folds')
axes[1].set_xlabel('Anzahl Folds')
axes[1].set_ylabel('Accuracy')
axes[1].set_xticks(fold_counts)
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Die Fehlerbalken zeigen Standardabweichungen, keine 95%-Konfidenzintervalle.")
print("Eine kleine Streuung in diesem Versuch beweist keine allgemein optimale Fold-Anzahl.")

## 6. Evaluate Model Performance

Jetzt testen wir, wie gut unser neuronales Netz auf ungesehenen Daten funktioniert.

In [ ]:
y_train_pred = mlp.predict(X_train_scaled)
y_test_pred = mlp.predict(X_test_scaled)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Model Performance:")
print(f"Training Accuracy: {train_accuracy:.4f} ({train_accuracy * 100:.2f}%)")
print(f"Test Accuracy:     {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Abstand Training minus Test: {train_accuracy - test_accuracy:+.4f}")
print("Ein Abstand kann auf Überanpassung hinweisen, beweist sie aber nicht.")
print("Auch ähnliche Werte garantieren keine gute Leistung auf allen neuen Daten.")

print(f"\n5-fach CV: Mittelwert {cv_mean:.4f}, Streuung {cv_std:.4f}")
print(f"Test Set: {test_accuracy:.4f}")
print("CV und Test sind unterschiedliche Schätzungen anhand verschiedener Daten.")
print("Die Fold-Streuung ist keine Toleranzgrenze für die Test-Accuracy.")

print("\nKlassifikation auf dem Test Set:")
print(classification_report(y_test, y_test_pred, target_names=iris.target_names))

In [ ]:
# Konfusionsmatrix visualisieren
plt.figure(figsize=(12, 5))

# Training Set Confusion Matrix
plt.subplot(1, 2, 1)
cm_train = confusion_matrix(y_train, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', 
           xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title(f'Training Set\nAccuracy: {train_accuracy:.3f}', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Test Set Confusion Matrix
plt.subplot(1, 2, 2)
cm_test = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
           xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title(f'Test Set\nAccuracy: {test_accuracy:.3f}', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.tight_layout()
plt.show()

# Fehleranalyse
test_errors = np.where(y_test != y_test_pred)[0]
if len(test_errors) > 0:
    print(f"\n🔍 Fehleranalyse: {len(test_errors)} falsche Vorhersagen im Test Set")
    for i in test_errors:
        actual = iris.target_names[y_test[i]]
        predicted = iris.target_names[y_test_pred[i]]
        print(f"Sample {i}: Actual={actual}, Predicted={predicted}")
else:
    print("\n🎉 Perfekte Klassifikation! Keine Fehler im Test Set.")

## 7. Vorhersagen für neue Blüten

Wir verwenden denselben Scaler wie beim Training. `predict_proba` liefert für jede
Klasse eine Modellwahrscheinlichkeit. Ein hoher Wert bedeutet, dass das Modell
diese Klasse stark bevorzugt, nicht dass die Vorhersage garantiert richtig ist.
Auch ein überzeugtes Modell kann falsch liegen.

In [ ]:
# Beispiel-Blüten für Vorhersagen definieren
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],  # Typische Setosa
    [6.9, 3.1, 4.9, 1.5],  # Typische Versicolor  
    [6.3, 3.3, 6.0, 2.5],  # Typische Virginica
    [5.8, 2.7, 4.1, 1.0]   # Grenzfall
])

# Wichtig: Neue Daten mit dem gleichen Scaler transformieren!
new_flowers_scaled = scaler.transform(new_flowers)

# Vorhersagen machen
predictions = mlp.predict(new_flowers_scaled)
probabilities = mlp.predict_proba(new_flowers_scaled)

print("🔮 Vorhersagen für neue Iris-Blüten:")
print("=" * 60)

feature_names = ['Sepal Length', 'Sepal Width', 'Petal Length', 'Petal Width']

for i, (flower, pred, probs) in enumerate(zip(new_flowers, predictions, probabilities)):
    print(f"\n🌸 Blüte {i+1}:")
    print(f"Features: {dict(zip(feature_names, flower))}")
    print(f"Vorhersage: {iris.target_names[pred]} (Klasse {pred})")
    print("Wahrscheinlichkeiten:")
    for j, (species, prob) in enumerate(zip(iris.target_names, probs)):
        print(f"  {species}: {prob:.3f} ({prob*100:.1f}%)")

    # Verteilung der Modellwahrscheinlichkeiten
    max_prob = np.max(probs)
    if max_prob > 0.9:
        confidence = "Hohe Modellwahrscheinlichkeit"
    elif max_prob > 0.7:
        confidence = "Überwiegend auf eine Klasse konzentriert"
    elif max_prob > 0.5:
        confidence = "Weniger eindeutig"
    else:
        confidence = "Auf mehrere Klassen verteilt"

    print(f"Modellausgabe: {confidence} (Max: {max_prob:.3f})")

In [ ]:
# Wahrscheinlichkeiten visualisieren
plt.figure(figsize=(12, 8))

# Subplot für jede neue Blüte
for i in range(len(new_flowers)):
    plt.subplot(2, 2, i+1)
    
    # Balkendiagramm der Wahrscheinlichkeiten
    bars = plt.bar(iris.target_names, probabilities[i], 
                  color=['red', 'green', 'blue'], alpha=0.7)
    
    # Vorhersage hervorheben
    predicted_idx = predictions[i]
    bars[predicted_idx].set_alpha(1.0)
    bars[predicted_idx].set_edgecolor('black')
    bars[predicted_idx].set_linewidth(2)
    
    plt.title(f'Blüte {i+1}: {iris.target_names[predicted_idx]}', fontweight='bold')
    plt.ylabel('Wahrscheinlichkeit')
    plt.ylim(0, 1)
    
    # Werte auf Balken anzeigen
    for bar, prob in zip(bars, probabilities[i]):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{prob:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Model-Insights anzeigen
print("\n🧠 Model-Insights:")
print(f"📊 Anzahl Parameter: {sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)}")
print(f"🏗️  Schichtgrößen: Input({mlp.coefs_[0].shape[0]}) → Hidden({mlp.coefs_[0].shape[1]}) → Hidden({mlp.coefs_[1].shape[1]}) → Output({mlp.coefs_[2].shape[1]})")
print(f"⚙️  Aktivierungsfunktion: {mlp.activation}")
print(f"🎯 Training-Iterationen: {mlp.n_iter_}")
print(f"📈 Final Training Loss: {mlp.loss_curve_[-1]:.6f}")

## Zusammenfassung

- Das Netz hat zwei versteckte Schichten mit 10 und 5 Neuronen.
- Training und Cross-Validation verwenden dieselben Modelleinstellungen.
- Die Skalierung wird nur aus den jeweiligen Trainingsdaten gelernt.
- Cross-Validation zeigt die Accuracy auf mehreren Aufteilungen. Mittelwert und
  Streuung helfen bei der Einordnung, sind aber keine Garantie für neue Daten.
- Der separate Test prüft das auf allen Trainingsdaten trainierte Hauptmodell.
- Bei wenigen Testblüten verändert schon ein einzelner Fehler die Accuracy sichtbar.
- Modellwahrscheinlichkeiten beschreiben die Vorhersage des Modells, nicht deren
  garantierte Zuverlässigkeit.

**Zum Ausprobieren:** Variiere `random_state` bei der Datenaufteilung und vergleiche
die Ergebnisse. Ändere Modelleinstellungen am Hauptmodell; `clone` übernimmt sie
automatisch für die Cross-Validation. Nutze den Test nicht zur Auswahl der Einstellungen.